# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display, clear_output
from openai import OpenAI
import json

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

# set up environment
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [ ]:
# Prompts
system_prompt = """You are a resourceful physicist with vast knowledge in physics and 
math who answers questions about physics and math.
Please do not answer questions that you do not know or are not sure about.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

question = """
Please explain quantum mechanics in a way that is easy to understand.
"""

In [ ]:
def messages_for():
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

In [ ]:
def get_topic_references(question_topic):
    return [
        "https://en.wikipedia.org/wiki/Physics",
        "https://en.wikipedia.org/wiki/Quantum_mechanics",
        "https://en.wikipedia.org/wiki/Quantum_field_theory",
        "https://en.wikipedia.org/wiki/Quantum_electrodynamics",
        "https://en.wikipedia.org/wiki/Quantum_chromodynamics",
        "https://en.wikipedia.org/wiki/Quantum_gravity",
        "https://en.wikipedia.org/wiki/Quantum_field_theory",
    ]


In [ ]:
# There's a particular dictionary structure that's required to describe our function:

question_topic_function = {
    "name": "get_topic_references",
    "description": "Take the topic of a technical question and responds with a list of references to relevant topics.",
    "parameters": {
        "type": "object",
        "properties": {
            "question_topic": {
                "type": "string",
                "description": "The topic of the technical question to answer.",
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": question_topic_function}]

In [ ]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_topic_references":
        arguments = json.loads(tool_call.function.arguments)
        topic = arguments.get('question_topic')
        topic_references = get_topic_references(topic)
        response = {
            "role": "tool",
            "content": topic_references,
            "tool_call_id": tool_call.id
        }
    return response

In [ ]:
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)

def question_answer(the_model, message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [
        {"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    
    response = openrouter.chat.completions.create(
        model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        response = openrouter.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [ ]:
import gradio as gr 

In [ ]:
gr.ChatInterface(fn=question_answer, type="messages").launch()